# Conecta+ · ¿A quién contactar y por qué se van?

Solución de la prueba técnica de Ciencia de Datos (nivel Senior). El notebook corre de principio a fin en
Google Colab: sube `datos_conecta.zip` al panel de archivos (o déjalo en tu Drive) y ejecuta *Entorno de ejecución → Ejecutar todo*.

**Cómo está organizado**

| Sección | Qué responde |
|---|---|
| 1. Datos y calidad | Qué hay en cada fuente, qué problemas tiene y cómo se resolvieron |
| 2. Etiquetas | Cómo se reconstruye el churn a 90 días y por qué se puede confiar en esa reconstrucción |
| 3. Texto de tickets | Cómo se aprovecha el texto (strands-decider + pysentimiento) y sus límites |
| 4. Variables | Qué sabe el modelo de cada cliente *a la fecha de corte* |
| 5. Modelos y MLflow | Validación temporal, 4 corridas y cuál se elige |
| 6. Efecto de los llamados | ¿Contaminan las etiquetas los contactos de retención? |
| 7. Decisión de contacto | A quién llamar, valor esperado y sensibilidad |
| 8. Hallazgos | Lo que la gerencia debería saber |
| 9. Producción | Cómo operarlo y qué vigilar |

La lógica está en el paquete `conecta/` (funciones probadas con `pytest`, carpeta `tests/`); este notebook orquesta y explica.

In [1]:
# Entorno: en Colab se clona el repositorio (para tener el paquete `conecta/`) y se ubica el zip de datos.
import os, sys, subprocess, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/unfresh25/conecta-churn.git"

if IN_COLAB:
    if not Path("/content/repo/conecta").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # Opción A: zip subido a /content. Opción B: descomentar para leerlo desde Drive.
    ZIP_PATH = Path("/content/datos_conecta.zip")
    # from google.colab import drive; drive.mount("/content/drive")
    # ZIP_PATH = Path("/content/drive/MyDrive/<ruta>/datos_conecta.zip")
else:
    ZIP_PATH = Path("datos_conecta.zip")

DATA_DIR = Path("datos")
if not (DATA_DIR / "conecta.db").exists():
    assert ZIP_PATH.exists(), f"No encuentro {ZIP_PATH}: sube datos_conecta.zip al entorno."
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
sys.path.insert(0, str(Path.cwd()))
print("Colab:", IN_COLAB, "| datos:", sorted(p.name for p in DATA_DIR.iterdir()))

Colab: False | datos: ['conecta.db', 'eventos.jsonl.gz', 'tickets.jsonl']


In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
SEED = 42
TODAY = pd.Timestamp("2025-12-31")

## 1. Datos y calidad

Tres sistemas distintos: la base relacional (`conecta.db`: clientes, suscripciones, facturas, etiquetas y clientes a puntuar),
la plataforma de atención (`tickets.jsonl`) y los logs de la app (`eventos.jsonl.gz`, solo 2025).
No hay diccionario de datos, así que cada supuesto sobre un campo queda escrito aquí.

In [3]:
from conecta.data import load_data, clean

raw = load_data(DATA_DIR)
data = clean(raw)
pd.DataFrame({
    "tabla": ["clientes", "suscripciones", "facturas", "etiquetas_entrenamiento", "clientes_a_puntuar", "tickets (registros)", "eventos"],
    "filas": [len(raw.clients), len(raw.subscriptions), len(raw.invoices), len(raw.labels), len(raw.to_score), len(raw.tickets_raw), len(raw.events)],
})

,tabla,filas
0,clientes,3000
1,suscripciones,3361
2,facturas,34138
3,etiquetas_entrenamiento,4693
4,clientes_a_puntuar,1556
5,tickets (registros),4452
6,eventos,103927


In [4]:
# Problemas encontrados y efecto de la limpieza (los conteos se recalculan con los datos)
reloads = len(raw.invoices) - raw.invoices["factura_id"].nunique()
quality = pd.DataFrame([
    ("ciudad: grafías distintas -> ciudades canónicas", f"{raw.clients['ciudad'].nunique()} -> {data.clients['ciudad'].nunique()}"),
    ("ciudad sin mapear (NaN)", int(data.clients["ciudad"].isna().sum())),
    ("edad inválida (0 o 999) -> NaN", int(data.clients["edad"].isna().sum())),
    ("facturas recargadas (mismo factura_id) eliminadas", reloads),
    ("cobros dobles reales (id distinto, mismo periodo) marcados", int(data.invoices["is_duplicate_charge"].sum())),
    ("facturas sin fecha_pago (impagas a hoy)", int(data.invoices["fecha_pago"].isna().sum())),
    ("eventos duplicados (event_id) eliminados", len(raw.events) - len(data.events)),
], columns=["problema", "valor"])
quality

,problema,valor
0,ciudad: grafías distintas -> ciudades canónicas,17 -> 6
1,ciudad sin mapear (NaN),0
2,edad inválida (0 o 999) -> NaN,65
3,facturas recargadas (mismo factura_id) eliminadas,677
4,"cobros dobles reales (id distinto, mismo perio...",699
5,facturas sin fecha_pago (impagas a hoy),1447
6,eventos duplicados (event_id) eliminados,1616


In [5]:
# Controles: si un conteo cambia, algo en la fuente o en la limpieza cambió y hay que investigarlo.
assert set(data.clients["ciudad"].dropna()) == {"Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"}
assert data.clients["ciudad"].isna().sum() == 0
assert data.clients["edad"].isna().sum() == 65
assert reloads == 677 and data.invoices["is_duplicate_charge"].sum() == 699
assert len(raw.events) - len(data.events) == 1616
print("Controles de calidad OK")

Controles de calidad OK


**Supuestos y decisiones sobre los campos**

- `estado_actual` describe al cliente *hoy* (31-dic-2025), no en cada fecha de corte: ningún cliente "activo" aparece como churn en las etiquetas.
  Usarlo sería una fuga de información, así que **no se usa como variable**.
- `contactado_por_retencion` / `fecha_contacto_retencion`: el 90 % de los contactos de los clientes etiquetados ocurrió *después* del corte.
  Solo se usan contactos con fecha ≤ corte.
- `fecha_carga` es cuándo la factura entró al sistema (1 a 4 días después de iniciado el periodo). Una factura solo "existe" para el modelo
  si se cargó antes del corte.
- Una factura con el mismo `factura_id` repetido es una recarga del sistema (misma factura, otra `fecha_carga`); con id distinto para el
  mismo cliente, periodo y suscripción es un **cobro doble real** (ver hallazgos).
- `fecha_pago` vacía = no pagada a hoy. Para un corte pasado, una factura pagada después del corte cuenta como impaga en ese corte.
- `motivo_cancelacion = cambio_plan` cierra una suscripción y abre otra: no es churn.

### 1.1 Tickets: dos esquemas en un mismo archivo

`tickets.jsonl` mezcla dos formatos de la plataforma de atención. El **A** (2023-2025) trae `customer: "CL-012375"`, fechas en UTC,
`category` y `text`. El **B** (2025) trae el cliente anidado (`cliente.id`), fechas con zona `-05:00`, un `tipo` con categoría y
subcategoría, y una conversación (`mensajes`) en la que habla el cliente y responde el agente.

Se unifican en un solo esquema: `cliente_id` entero, fechas en hora de Bogotá, `text` = primer mensaje del **cliente**
(los mensajes del agente son 3 frases fijas y no aportan) y `status` abierto/cerrado.

In [6]:
from conecta.tickets import unify_tickets

data.tickets, dropped = unify_tickets(data.tickets_raw)
print(f"{len(data.tickets_raw)} registros -> {len(data.tickets)} tickets | descartados (cliente o fecha ilegible): {dropped}")
print("por esquema:", data.tickets["schema"].value_counts().to_dict())
print("textos únicos de cliente:", data.tickets["text"].nunique())
assert data.tickets["schema"].value_counts().to_dict() == {"A": 2553, "B": 1899} and dropped == 0

4452 registros -> 4452 tickets | descartados (cliente o fecha ilegible): 0
por esquema: {'A': 2553, 'B': 1899}
textos únicos de cliente: 38
